# Data Ingestion — Getting Started

The ingestion pipeline is the primary entry point for loading geospatial data into
DynaStore collections. It is exposed as an **OGC API - Processes (Part 1)** process
and follows a fully asynchronous, task-based architecture.

## How it works

```
POST /catalogs/{cat}/collections/{coll}/processes/ingestion/execution
  │
  ├─ 1. Validate: process scope matches the URL mount (collection-scoped)
  ├─ 2. Inject catalog_id / collection_id from URL path into inputs
  ├─ 3. Validate: collection exists, has a physical LayerConfig
  ├─ 4. Create task record (PENDING) in tasks.tasks
  ├─ 5. Return 201 Created with Location header for polling
  │
  Dispatcher claims task (FOR UPDATE SKIP LOCKED)
  │
  ├─ 6. Resolve or create asset from URI / asset_id
  ├─ 7. Open source file (Fiona) and stream records in batches
  ├─ 8. Map columns → geometry + attributes + identity
  ├─ 9. Upsert batches to all WRITE-routed drivers
  ├─10. Recalculate collection extents
  └─11. Mark task COMPLETED (or FAILED)
```

## Process scope

The `ingestion` process declares `scopes=[collection]`. This means it can **only**
be executed at the collection-scoped URL:

```
POST /catalogs/{catalog_id}/collections/{collection_id}/processes/ingestion/execution
```

Posting to the unscoped `/processes/ingestion/execution` or the catalog-only
`/catalogs/{catalog_id}/processes/ingestion/execution` returns **400 Bad Request**
with a message pointing to the valid route. The `catalog_id` and `collection_id`
come from the URL path — putting them in the request body is an error.

## Task lifecycle

| Status | Meaning |
|--------|---------|
| `PENDING` | Queued, waiting for a runner to claim |
| `ACTIVE` | Claimed by a runner, heartbeat expected every 30s |
| `COMPLETED` | Successfully finished |
| `FAILED` | Error encountered, may retry with exponential backoff |
| `DEAD_LETTER` | Max retries exhausted, requires manual intervention |

This notebook walks through: process discovery, submission, monitoring, and verification.

In [ ]:
# === Parameters ===
# Adjust these to point at your DynaStore instance.
# For the on-premise 'catalog' service (no auth required), use port 82.

import os
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"   # API base URL
CATALOG_ID = "demo-ingestion"    # Will be created by this notebook
COLLECTION_ID = "crop-stations"  # Target collection for ingestion

In [ ]:
import httpx
import asyncio
import json

client = httpx.AsyncClient(base_url=BASE, timeout=60)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

# Create demo catalog
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Ingestion Demo", "description": "Ingestion Demo catalog."})
_check(r, "Catalog")

# Create collection with a physical layer config (required for ingestion)
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Crop Monitoring Stations",
    "description": "Demo collection for ingestion tutorial",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")
# Wait for provisioning to complete before creating collections
for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("Warning: catalog still provisioning after 30s")

---
## 1. Process discovery (HATEOAS)

Before hitting `/execution` directly, discover the process and its allowed
execution URLs. DynaStore advertises an OGC-compliant `rel=execute` link
(relation `http://www.opengis.net/def/rel/ogc/1.0/execute`) on every process
description — one per declared scope, as a templated URL.

A client can therefore:

1. `GET /processes/{process_id}` to read the scopes, input schema, and the
   templated execute URL(s);
2. substitute its `catalog_id` / `collection_id` and POST the body.

The scoped listing endpoint is even simpler — it returns only processes that
can run at that mount, with concrete (non-templated) execute URLs.

In [ ]:
# Discover the ingestion process: scopes + execute link(s).
r = await client.get("/processes/ingestion")
_check(r, "Process description")
doc = r.json()

print("scopes:", doc.get("scopes"))
print("jobControlOptions:", doc.get("jobControlOptions"))
print()

EXECUTE_REL = "http://www.opengis.net/def/rel/ogc/1.0/execute"
execute_templates = [l for l in doc.get("links", []) if l.get("rel") == EXECUTE_REL]
print(f"Found {len(execute_templates)} rel=execute link(s):")
for l in execute_templates:
    print(f"  - {l['href']}  (templated={l.get('templated')}, method={l.get('method')})")

# Scoped listing: filtered to processes executable at this mount,
# with concrete (non-templated) execute URLs.
r = await client.get(f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes")
_check(r, "Scoped process list")
processes = r.json().get("processes", [])
print()
print(f"Processes executable at /catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}:")
for p in processes:
    exec_link = next((l for l in p.get("links", []) if l.get("rel") == EXECUTE_REL), None)
    print(f"  - {p['id']} (scopes={p.get('scopes')}) -> {exec_link['href'] if exec_link else '(no execute link)'}")

---
## 2. CSV Ingestion with lat/lon columns

The most common scenario: a CSV file with latitude and longitude columns.
The `column_mapping` tells the ingestion engine how to extract geometry
from the source columns.

### Key fields in `column_mapping`

| Field | Purpose |
|-------|---------|
| `external_id` | Source column used as unique identity for conflict resolution |
| `csv_lat_column` | Column containing latitude |
| `csv_lon_column` | Column containing longitude |
| `csv_elevation_column` | Optional: column with elevation (creates 3D points) |
| `attributes_source_type` | `"all"` (default) or `"explicit_list"` |

The execute URL — the `href` of the `rel=execute` link from the previous cell,
with `{catalog_id}` / `{collection_id}` substituted:

```
POST /catalogs/{catalog_id}/collections/{collection_id}/processes/ingestion/execution
```

### Body shape

The request body is an OGC `ExecuteRequest` with a single input,
`ingestion_request`:

```json
{
  "inputs": {
    "ingestion_request": { ...TaskIngestionRequest fields... }
  }
}
```

> **Note:** Do **not** put `catalog_id` or `collection_id` inside `inputs` —
> the URL path is authoritative and a conflict with the body is rejected
> (HTTP 400). The server injects them from the path into `inputs` before the
> task runs, so every task handler sees a uniform shape.

In [ ]:
# Build the ingestion request body. Note: no catalog_id / collection_id —
# they come from the URL path and including them in the body is rejected.
ingestion_body = {
    "inputs": {
        "ingestion_request": {
            "asset": {"uri": "/data/crop_stations.csv"},
            "encoding": "utf-8",
            "column_mapping": {
                "external_id": "station_id",
                "csv_lat_column": "latitude",
                "csv_lon_column": "longitude",
                "csv_elevation_column": "elevation_m",
                "attributes_source_type": "all",
            },
            "source_srid": 4326,
        }
    }
}

print("Ingestion request body:")
print(json.dumps(ingestion_body, indent=2))

# The concrete execute URL — equivalent to substituting the template from cell 1.
exec_url = f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution"

# To actually submit (requires a real file at the URI), use async (Prefer: respond-async):
#
# r = await client.post(exec_url, json=ingestion_body, headers={"Prefer": "respond-async"})
# assert r.status_code == 201, r.text        # 201 Created + Location header
# status_info = r.json()
# job_id = status_info["jobID"]
# status_url = r.headers.get("Location")     # canonical polling URL
# print("Job ID:", job_id, "| polling at:", status_url)

### Scope validation errors

Two failure modes are worth knowing, since both return **400 Bad Request**
**before** any task is written to the database (so you never produce a
stale PENDING row from a misrouted call):

1. **Wrong URL mount.** POSTing a collection-scoped process at an unscoped
   or catalog-only URL — the error message names the valid route.
2. **Body / URL conflict.** Including `catalog_id` or `collection_id` in
   `inputs` with a different value than the URL path — the URL wins and
   the server refuses the request.

In [ ]:
# 1) Wrong URL mount — posting the collection-scoped "ingestion" process at
#    the unscoped endpoint. Should return 400 with a scope-hint message.
r = await client.post("/processes/ingestion/execution", json=ingestion_body)
print(f"Wrong URL:        {r.status_code}")
assert r.status_code == 400, r.text
print(f"  detail: {r.json().get('detail')}")

# 2) Body/URL conflict — include a catalog_id in inputs that doesn't match
#    the URL path. Should return 400 with a conflict message.
conflicting_body = {
    "inputs": {
        "catalog_id": "not-the-real-catalog",
        "ingestion_request": ingestion_body["inputs"]["ingestion_request"],
    }
}
r = await client.post(exec_url, json=conflicting_body)
print(f"Body conflict:    {r.status_code}")
assert r.status_code == 400, r.text
print(f"  detail: {r.json().get('detail')}")

---
## 3. Inline feature ingestion (no file needed)

For testing and small datasets, you can ingest features directly via the
STAC/OGC Features API without going through the task system.
This is synchronous and immediate.

In [ ]:
# Direct feature ingestion via STAC Items endpoint (synchronous)
features = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "station-001",
            "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
            "bbox": [12.49, 41.89, 12.49, 41.89],
            "properties": {
                "name": "Roma Fiumicino",
                "crop_type": "wheat",
                "yield_t_ha": 4.2,
                "year": 2024
            }
        },
        {
            "type": "Feature",
            "id": "station-002",
            "geometry": {"type": "Point", "coordinates": [11.25, 43.77]},
            "bbox": [11.25, 43.77, 11.25, 43.77],
            "properties": {
                "name": "Firenze Peretola",
                "crop_type": "olive",
                "yield_t_ha": 2.8,
                "year": 2024
            }
        },
        {
            "type": "Feature",
            "id": "station-003",
            "geometry": {"type": "Point", "coordinates": [9.19, 45.46]},
            "bbox": [9.19, 45.46, 9.19, 45.46],
            "properties": {
                "name": "Milano Linate",
                "crop_type": "rice",
                "yield_t_ha": 6.1,
                "year": 2024
            }
        }
    ]
}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=features,
)
_check(r, "Ingest")

---
## 4. Column mapping modes

The `column_mapping.attributes_source_type` controls which source columns
become feature properties.

### `"all"` (default)
All source columns except geometry and identity columns become properties.
Quick and easy for exploratory work.

### `"explicit_list"`
Only columns listed in `attribute_mapping` are included. Each entry maps
a `source` column to a `map_to` target name. You can also inject `constant`
values (e.g., a fixed data source tag).

```json
{
    "column_mapping": {
        "external_id": "station_id",
        "csv_lat_column": "lat",
        "csv_lon_column": "lon",
        "attributes_source_type": "explicit_list",
        "attribute_mapping": [
            {"source": "station_name", "map_to": "name"},
            {"source": "crop",         "map_to": "crop_type"},
            {"source": "yield_kg",     "map_to": "yield_t_ha"},
            {"constant": "FAO",         "map_to": "data_source"}
        ]
    }
}
```

Use `explicit_list` when:
- Source columns have inconsistent names across files
- You need to rename columns to match your schema
- You want to inject constant metadata values
- You need to exclude sensitive or irrelevant columns

---
## 5. Task monitoring

When ingestion runs asynchronously (default when `Prefer: respond-async`
is set, or when `jobControlOptions` contains `async-execute`), the response
is **201 Created** with a `Location` header carrying the canonical job
status URL. Poll that URL to track progress.

The OGC job moves through: `accepted` → `running` → `successful` (or
`failed`, `dismissed`). While `running`, the `progress` field reports the
percentage of records processed.

In [ ]:
# Poll an OGC job at its Location URL until a terminal state.
async def poll_job(status_url: str, interval: int = 2, max_wait: int = 120):
    """Poll an OGC StatusInfo until terminal state (successful/failed/dismissed)."""
    elapsed = 0
    while elapsed < max_wait:
        r = await client.get(status_url)
        if r.status_code != 200:
            print(f"Error polling job: {r.status_code} — {r.text[:200]}")
            return None
        info = r.json()
        status = info.get("status", "unknown")
        progress = info.get("progress", 0)
        print(f"  [{elapsed}s] status={status} progress={progress}%")
        if status in ("successful", "failed", "dismissed"):
            return info
        await asyncio.sleep(interval)
        elapsed += interval
    print("Timeout waiting for job completion")
    return None

# Example — paste the Location header from an async POST response:
# status_url = r.headers["Location"]
# final = await poll_job(status_url)
# print(json.dumps(final, indent=2))
print("poll_job() defined — feed it the `Location` header from an async execute response")

---
## 6. Verify ingested data

After ingestion completes, query the collection via the STAC Items endpoint
to confirm features were stored correctly.

In [ ]:
# List items in the collection
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items")
items = r.json()

print(f"Total features: {items.get('numberMatched', len(items.get('features', [])))}")
for feat in items.get("features", []):
    props = feat.get("properties", {})
    geom = feat.get("geometry", {})
    coords = geom.get("coordinates", [])
    print(f"  {feat.get('id')}: {props.get('name')} @ {coords} — {props.get('crop_type')} {props.get('yield_t_ha')} t/ha")

---
## 7. Batch tuning

For large files, tune the batch parameters to balance memory usage and throughput.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `database_batch_size` | 500 | Records per database transaction |
| `max_batch_memory_mb` | 100 | Memory threshold before flushing (takes precedence) |
| `read_batch_size` | 1000 | Records to read from source per chunk |
| `offset` | 0 | Skip N records from the start |
| `limit` | None | Process at most N records |

### Guidelines

- **Small files (<10K records)**: defaults work fine
- **Large files (>100K records)**: increase `database_batch_size` to 2000–5000
- **Memory-constrained**: lower `max_batch_memory_mb` to 50
- **Resume after failure**: use `offset` to skip already-ingested records
- **Sampling**: use `limit` to ingest a subset for testing

```json
{
    "inputs": {
        "ingestion_request": {
            "database_batch_size": 2000,
            "max_batch_memory_mb": 50,
            "read_batch_size": 5000,
            "offset": 0,
            "limit": 10000
        }
    }
}
```

---
## 8. Geometry formats

The ingestion engine supports multiple geometry input formats:

| Format | Column mapping field | Use case |
|--------|---------------------|----------|
| Lat/Lon columns | `csv_lat_column` + `csv_lon_column` | CSV with separate coordinate columns |
| WKT | `csv_wkt_column` | CSV with Well-Known Text geometry |
| WKB | `geometry_wkb` | Parquet/binary with Well-Known Binary |
| Native GeoJSON | (automatic) | GeoJSON/Shapefile via Fiona — geometry extracted automatically |

### WKT example

```json
{
    "column_mapping": {
        "external_id": "parcel_id",
        "csv_wkt_column": "geom_wkt",
        "attributes_source_type": "all"
    }
}
```

### GeoJSON/Shapefile

For files that Fiona reads natively (`.geojson`, `.shp`, `.gpkg`), geometry
is extracted automatically from the `geometry` field. No column mapping needed
for geometry — just set `external_id` and attributes.

In [ ]:
# Cleanup
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()